In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q yt-dlp ultralytics boxmot

# torchreid must be installed from source (PyPI package is broken)
import os, sys
if not os.path.isdir('/kaggle/working/deep-person-reid/.git'):
    !git clone -q https://github.com/KaiyangZhou/deep-person-reid.git /kaggle/working/deep-person-reid
!pip install -q -e /kaggle/working/deep-person-reid/

# pip install -e doesn't hot-reload in the current kernel — add path manually
if '/kaggle/working/deep-person-reid' not in sys.path:
    sys.path.insert(0, '/kaggle/working/deep-person-reid')

import importlib, torchreid
importlib.invalidate_caches()
print('torchreid OK:', torchreid.__version__)


In [ ]:
# ── Cell 2: Clone repo ─────────────────────────────────────────────────────────
# NOTE: REPO_ROOT must NOT be /kaggle/working/ai-football-t4 —
# Kaggle pre-creates that directory for the notebook file itself,
# so git clone silently fails and src/ never appears.
import os, shutil

REPO_ROOT = '/kaggle/working/repo'

if os.path.isdir(f'{REPO_ROOT}/.git'):
    print('Repo already present, pulling latest...')
    !git -C {REPO_ROOT} pull
else:
    if os.path.exists(REPO_ROOT):
        shutil.rmtree(REPO_ROOT)  # remove any partial/empty directory
    !git clone https://github.com/jordiagi/ai-football-t4 {REPO_ROOT}

print('Repo root:', REPO_ROOT)
print('src/ contents:', os.listdir(f'{REPO_ROOT}/src'))


In [ ]:
# ── Cell 3: Download video ─────────────────────────────────────────────────────
# Video is saved to /kaggle/working/downloads/ which persists across cell re-runs
# within the same session (survives git pull / repo re-clone in cell 2).
import os, glob
REPO_ROOT = '/kaggle/working/repo'
DOWNLOADS_DIR = '/kaggle/working/downloads'
VIDEO_URL = 'https://youtu.be/Flm3Ki1TCXY'

existing = glob.glob(f'{DOWNLOADS_DIR}/*.mp4')
if existing:
    VIDEO_PATH = existing[0]
    print(f'Video already downloaded: {VIDEO_PATH}')
else:
    print('Downloading video...')
    !python {REPO_ROOT}/src/download.py "{VIDEO_URL}" --output-dir {DOWNLOADS_DIR}
    existing = glob.glob(f'{DOWNLOADS_DIR}/*.mp4')
    VIDEO_PATH = existing[0] if existing else ''
    print(f'Downloaded: {VIDEO_PATH}')


In [ ]:
# ── Cell 4: Verify reference images ───────────────────────────────────────────
from pathlib import Path
REPO_ROOT = '/kaggle/working/repo'

ref_images = [
    Path(REPO_ROOT) / 'data/player_refs/player1.jpg',
    Path(REPO_ROOT) / 'data/player_refs/player2.jpg',
]
for r in ref_images:
    assert r.exists(), f'Missing reference image: {r}'
    print('Reference image OK:', r)


In [ ]:
# ── Cell 5: Run pipeline ───────────────────────────────────────────────────────
# VIDEO_PATH is set in cell 3 — no re-download needed if already done this session
REPO_ROOT = '/kaggle/working/repo'
!python {REPO_ROOT}/src/pipeline.py \
    --video-path "{VIDEO_PATH}" \
    --ref-images {REPO_ROOT}/data/player_refs/player1.jpg {REPO_ROOT}/data/player_refs/player2.jpg \
    --output-dir /kaggle/working/output


In [ ]:
# ── Cell 6: Preview output videos ─────────────────────────────────────────────
OUTPUT_DIR = '/kaggle/working/output'
from IPython.display import Video, display
display(Video(f'{OUTPUT_DIR}/highlight_short.mp4', embed=True))
display(Video(f'{OUTPUT_DIR}/annotated_full.mp4', embed=True))


In [ ]:
# ── Cell 7: Bundle output for download ────────────────────────────────────────
OUTPUT_DIR = '/kaggle/working/output'
!zip -r /kaggle/working/output_bundle.zip {OUTPUT_DIR}
from IPython.display import FileLink
FileLink('/kaggle/working/output_bundle.zip')
